# <center> Module 6: Recommender Systems</center>
## <center> Collaborative Filtering: Model Based Recommenders <br> MovieLens Movies Database </center>
<center>by: Nicole Woodland, P. Eng. for RoboGarden Inc. </center>

---

#### Ask the Right Question:

In this example, we have a database of Users and the ratings they have given to various movies. <br> Can we use this data to make movies recommendations to the user?

### <font color='blue'> Download the Dataset </font>
Download the Dataset from the following link: <br>
https://www.kaggle.com/rounakbanik/the-movies-dataset?select=movies_metadata.csv

In [3]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

There are a few steps to install SciKit-Surprise:
1. Install VS Code from: https://code.visualstudio.com/docs/setup/windows
2. Install the dependencies for C++ builder from: https://visualstudio.microsoft.com/visual-cpp-build-tools/
3. Run the pip command line below by uncommenting the pip install command
4. May need to reversion Python to 3.10 and numpy to 1.9?
   

In [5]:
#pip install scikit-surprise

<font color = "blue"> Read the dataset into the Pandas Data Frame!

In [11]:
import pandas as pd
#Load the small dataset
ratings = pd.read_csv("../ratings.csv")
#ratings = pd.read_csv("ratings_small.csv")

In [15]:
ratings.head(30)

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556
5,1,1968,4.0,1425942148
6,1,2762,4.5,1425941300
7,1,2918,5.0,1425941593
8,1,2959,4.0,1425941601
9,1,4226,4.0,1425942228


In [5]:
mov_ids = ratings.movieId.unique()
print(mov_ids)
print(len(mov_ids))


[   110    147    858 ... 165649 171051 171221]
45115


In [10]:
ratings.tail(10)

,userId,movieId,rating,timestamp
26024279,270896,54286,4.5,1257031701
26024280,270896,54503,4.0,1257033886
26024281,270896,55820,5.0,1257031660
26024282,270896,56174,3.5,1257034085
26024283,270896,56367,4.5,1257031529
26024284,270896,58559,5.0,1257031564
26024285,270896,60069,5.0,1257032032
26024286,270896,63082,4.5,1257031764
26024287,270896,64957,4.5,1257033990
26024288,270896,71878,2.0,1257031858


## Create a Model-Based Collaborative Filtering Recommender

In [17]:
ratings = ratings.drop('timestamp', axis = 1)

In [19]:
ratings

,userId,movieId,rating
0,1,110,1.0
1,1,147,4.5
2,1,858,5.0
3,1,1221,5.0
4,1,1246,5.0
...,...,...,...
26024284,270896,58559,5.0
26024285,270896,60069,5.0
26024286,270896,63082,4.5
26024287,270896,64957,4.5


In [14]:
ratings.iloc[:,0:1].values

array([[     1],
       [     1],
       [     1],
       ...,
       [270896],
       [270896],
       [270896]], dtype=int64)

### Import Surprise, and define data and and training and test sets

In [21]:
from surprise import Dataset, Reader
reader = Reader()

#Columns in df must be in the order profile_ID, Item_ID, Item_rating
data = Dataset.load_from_df(ratings, reader)

In [22]:
data

In [25]:
from surprise.model_selection import train_test_split
trainset, testset = train_test_split(data, test_size=0.25)

In [29]:
# Train the model - aka, guess the ratings users would make by determining the best case latent matrices for users and items
from surprise import SVD, accuracy
svd = SVD()
svd.fit(trainset)

In [30]:
# Check model performance:
trainset_predictions = [
    (trainset.to_raw_uid(uid), trainset.to_raw_iid(iid), rating)
    for uid, iid, rating in trainset.all_ratings()]
predictions = svd.test(trainset_predictions)
print("Training score:",accuracy.mse(predictions))

## THis part is typical process:
predictions = svd.test(testset)
print("Testing Score:",accuracy.mse(predictions))

MSE: 0.4585
Training score: 0.45845845111808425
MSE: 0.6398
Testing Score: 0.6398397531015764


This corresponds to 0.8 out of 5 units of error across the board. In the general sense, this is 'good enough' to give an idea if a user will have a positive or negative reaction to an item. Especially in a dataset where there is a high degree of subjectiveness in the data collection. The training score and tresting score are also similar, so the model should be reasonably fit.

In [20]:
predictions

[Prediction(uid=57847, iid=1674, r_ui=4.5, est=3.321250785979138, details={'was_impossible': False}),
 Prediction(uid=14793, iid=4878, r_ui=4.0, est=3.9183541887591, details={'was_impossible': False}),
 Prediction(uid=4480, iid=253, r_ui=4.5, est=3.7508305076744533, details={'was_impossible': False}),
 Prediction(uid=90203, iid=2496, r_ui=2.5, est=2.017369980049548, details={'was_impossible': False}),
 Prediction(uid=102651, iid=103141, r_ui=2.0, est=2.649006729539745, details={'was_impossible': False}),
 Prediction(uid=188851, iid=58975, r_ui=3.5, est=3.203356454613935, details={'was_impossible': False}),
 Prediction(uid=169074, iid=131160, r_ui=4.0, est=3.6033542413549218, details={'was_impossible': False}),
 Prediction(uid=260339, iid=73017, r_ui=4.5, est=4.11308817685027, details={'was_impossible': False}),
 Prediction(uid=220375, iid=508, r_ui=4.0, est=3.9344446644789164, details={'was_impossible': False}),
 Prediction(uid=24905, iid=2502, r_ui=4.0, est=3.6354969736075984, details

In [21]:
#Predict the rating of User 10 for Movie ID 34. (The rating is the 4th element in the array for the chosen user)
prediction = svd.predict(10,34)[3]  
print(prediction)

4.273112860672884


In [37]:
user_rating = []

for i in mov_ids:
    predict = svd.predict(10,i)[3]
    user_rating.append(predict)

In [39]:
user_rating

[4.297400028649263,
 3.8344876957879985,
 4.4664673849285865,
 4.298657126380337,
 4.113894198517546,
 4.196496205728968,
 4.619436140221071,
 4.354923908440194,
 4.429357589910895,
 4.32602414010134,
 4.216744510067108,
 3.77572773026816,
 4.2885659652459935,
 4.178900976415933,
 4.593743005569792,
 4.298233439471123,
 4.120953279108236,
 4.2548337175353925,
 4.314782486250392,
 4.205684296962398,
 3.896380581537777,
 4.135449341889914,
 3.844664767697621,
 4.183042078922433,
 4.289666691375968,
 4.335188928341973,
 4.250604771828811,
 3.504205024226399,
 3.5516899033296356,
 3.843193081466808,
 4.004866300737615,
 3.2232207079974846,
 3.606435364226549,
 3.6052241294862264,
 4.346108828684574,
 4.052514399120412,
 3.987784055437907,
 3.7023291346871843,
 4.109498074294657,
 3.9683908328490936,
 2.8183358781930554,
 4.055280426493969,
 3.8020597861992402,
 3.974003132141661,
 4.302065527547444,
 3.887575001885009,
 3.8297280042614537,
 3.1456198201684096,
 3.616744612073713,
 3.826021

In [41]:
len(user_rating)

45115

In [43]:
df_recommend = pd.DataFrame(list(zip(mov_ids, 
                                     user_rating)), columns = ["movie_id","Ranking"])  

In [45]:
df_recommend

,movie_id,Ranking
0,110,4.297400
1,147,3.834488
2,858,4.466467
3,1221,4.298657
4,1246,4.113894
...,...,...
45110,159050,3.849307
45111,159053,3.819775
45112,165649,3.819775
45113,171051,3.805559


In [47]:
df_recommend = df_recommend.sort_values(by=['Ranking'], ascending=False,ignore_index=True)

In [49]:
df_recommend.head(30)

,movie_id,Ranking
0,174053,4.857627
1,170705,4.805146
2,98491,4.734298
3,159817,4.731033
4,318,4.713420
5,171011,4.688050
6,120625,4.685690
7,73881,4.673063
8,31658,4.672746
9,2571,4.671364


In [29]:
df_recommend['movie_id'][0]

7153

In [30]:
grab = []
for i in range(0,5):
    grab.append(df_recommend["movie_id"][i].astype('str'))

print(grab)

['7153', '318', '170705', '142115', '100553']


In [31]:
# If the datasets matched up with the previous file, (or you had movies names with their ID keys)
# could use the index numbers to grab the movie names from the original Metadata dataset

print(df_movie_names["title"][df[df["id"]=="7153"].index])
print(df_movie_names["title"][df[df["id"]=="318"].index])

NameError: name 'df_movie_names' is not defined